In [1]:
import os
os.chdir(r"D:\GITHUB\PI2026")
print(os.getcwd())


D:\GITHUB\PI2026


In [3]:
import sys, subprocess

# Instala dependencias en el MISMO Python del notebook
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "folium", "branca", "openpyxl"], check=True)

print("Dependencias OK")


Dependencias OK


In [4]:
# Regenerar EXCELS de ZonasEscolares desde Resumen/ZonasEscolares_resumen.xlsx
# y luego volver a generar mapas (solo ZonasEscolares)

from pathlib import Path
import pandas as pd
import subprocess
import sys

ROOT = Path(".").resolve()
RESUMEN_XLSX = ROOT / "Resumen" / "ZonasEscolares_resumen.xlsx"
CATALOGO_MUNI = ROOT / "Data" / "municipalidades_catalog.csv"
OUT_EXCELS = ROOT / "ZonasEscolares" / "excels"
OUT_MAPS = ROOT / "ZonasEscolares" / "maps"

LIMPIAR_ANTERIORES = True  # True = borra excels/html previos para evitar archivos obsoletos

assert RESUMEN_XLSX.exists(), f"No existe: {RESUMEN_XLSX}"
assert CATALOGO_MUNI.exists(), f"No existe: {CATALOGO_MUNI}"

# 1) Cargar resumen editado
df = pd.read_excel(RESUMEN_XLSX, dtype=str).fillna("")
if "ubigeo_gestor" not in df.columns:
    raise KeyError("El resumen debe tener columna 'ubigeo_gestor'.")

# Normaliza ubigeo a 6 dígitos
df["ubigeo_gestor"] = (
    df["ubigeo_gestor"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"\D", "", regex=True)
    .str.zfill(6)
    .str[:6]
)

# 2) Catálogo para naming (slug) y admin oficial
cat = pd.read_csv(CATALOGO_MUNI, dtype=str).fillna("")
if "ubigeo" not in cat.columns:
    raise KeyError("municipalidades_catalog.csv debe tener columna 'ubigeo'.")
cat["ubigeo"] = (
    cat["ubigeo"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"\D", "", regex=True)
    .str.zfill(6)
    .str[:6]
)
for c in ["departamento", "provincia", "distrito", "slug"]:
    if c not in cat.columns:
        cat[c] = ""

cat_by_u = cat.drop_duplicates("ubigeo").set_index("ubigeo")

# 3) Preparar carpetas y limpiar salidas anteriores
OUT_EXCELS.mkdir(parents=True, exist_ok=True)
OUT_MAPS.mkdir(parents=True, exist_ok=True)

if LIMPIAR_ANTERIORES:
    for p in OUT_EXCELS.glob("*.xlsx"):
        p.unlink(missing_ok=True)
    for p in OUT_MAPS.glob("*.html"):
        p.unlink(missing_ok=True)

# 4) Exportar un Excel por ubigeo_gestor
resumen_rows = []
for u6, g in df.groupby("ubigeo_gestor", dropna=True):
    g = g.copy()

    if u6 in cat_by_u.index:
        dep = str(cat_by_u.at[u6, "departamento"]).replace("_", " ")
        prov = str(cat_by_u.at[u6, "provincia"]).replace("_", " ")
        dist = str(cat_by_u.at[u6, "distrito"]).replace("_", " ")
        slug = str(cat_by_u.at[u6, "slug"]).strip() or u6
    else:
        dep = prov = dist = ""
        slug = u6

    # Asegura columnas admin (sobrescribe con oficial si existe)
    g["departamento"] = dep
    g["provincia"] = prov
    g["distrito"] = dist

    # Orden sugerido
    first_cols = ["ubigeo_gestor", "departamento", "provincia", "distrito"]
    cols = [c for c in first_cols if c in g.columns] + [c for c in g.columns if c not in first_cols]
    g = g[cols]

    out_xlsx = OUT_EXCELS / f"{slug}.xlsx"
    g.to_excel(out_xlsx, index=False)

    resumen_rows.append({
        "ubigeo_gestor": u6,
        "archivo_rel": str(out_xlsx.relative_to(ROOT)),
        "n_filas": len(g),
    })

resumen_df = pd.DataFrame(resumen_rows).sort_values("ubigeo_gestor")
resumen_df.to_csv(OUT_EXCELS / "_resumen_excels_por_muni.csv", index=False, encoding="utf-8")
print(f"Excels generados: {len(resumen_df)}")

# 5) Regenerar mapas SOLO de ZonasEscolares
cmd = [sys.executable, str(ROOT / "maps_colegios.py"), "--excels-dir", str(OUT_EXCELS), "--out-dir", str(OUT_MAPS), "--mode", "zonas"]
print("Ejecutando:", " ".join(cmd))
subprocess.run(cmd, cwd=ROOT, check=True)

print("Listo. Mapas regenerados en:", OUT_MAPS)


Excels generados: 188
Ejecutando: c:\Users\victo\AppData\Local\Python\pythoncore-3.14-64\python.exe D:\GITHUB\PI2026\maps_colegios.py --excels-dir D:\GITHUB\PI2026\ZonasEscolares\excels --out-dir D:\GITHUB\PI2026\ZonasEscolares\maps --mode zonas
Listo. Mapas regenerados en: D:\GITHUB\PI2026\ZonasEscolares\maps
